In [1]:
import sys
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (StandardScaler, OneHotEncoder, TargetEncoder,
                                   FunctionTransformer)
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

random_state = 42
best_x = 7          # chosen in 02_preprocessing
data_dir = "/Users/shouryakulkarni/IDX-Exchange-Data-Science"
pd.set_option('display.max_columns', 50)

print("python", sys.version.split()[0], "| pandas", pd.__version__,
      "| numpy", np.__version__, "| scikit-learn", sklearn.__version__)

python 3.12.12 | pandas 3.0.0 | numpy 2.4.2 | scikit-learn 1.8.0


In [2]:
clean = pd.read_csv(f"{data_dir}/cleaned_sfr.csv",
                    dtype={"PostalCode": str}, parse_dates=["CloseDate"])
print(clean.shape)
clean["CloseMonth"].value_counts().sort_index()

(107416, 20)


CloseMonth
2025-07    12071
2025-08    11406
2025-09    11422
2025-10    11995
2025-11     9704
2025-12    10424
2026-01     7460
2026-02     8938
2026-03    11541
2026-04    12455
Name: count, dtype: int64

In [3]:
target = "ClosePrice"

numeric_features = [
    "LivingArea", "BedroomsTotal", "BathroomsTotalInteger", "LotSqFt",
    "YearBuilt", "Latitude", "Longitude", "GarageSpaces", "Stories",
]
skewed_features = ["LivingArea", "LotSqFt"]
other_numeric = [c for c in numeric_features if c not in skewed_features]
binary_features = ["PoolPrivateYN", "ViewYN", "FireplaceYN", "NewConstructionYN"]
low_card_categorical = ["CountyOrParish"]
high_card_categorical = ["City", "PostalCode"]

features = numeric_features + binary_features + low_card_categorical + high_card_categorical

In [13]:
def fit_outlier_bounds(train, lo=0.005, hi=0.995):
    ppsf = train[target] / train["LivingArea"]
    return {
        "price_lo": train[target].quantile(lo), "price_hi": train[target].quantile(hi),
        "ppsf_lo": ppsf.quantile(lo),           "ppsf_hi": ppsf.quantile(hi),
    }

def apply_outlier_bounds(data, b):
    ppsf = data[target] / data["LivingArea"]
    keep = (data[target].between(b["price_lo"], b["price_hi"]) &
            ppsf.between(b["ppsf_lo"], b["ppsf_hi"]))
    return data[keep]

def make_preprocessor():
    skewed = Pipeline([
        ("impute", SimpleImputer(strategy="median", add_indicator=True)),
        ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
        ("scale", StandardScaler()),
    ])
    numeric = Pipeline([
        ("impute", SimpleImputer(strategy="median", add_indicator=True)),
        ("scale", StandardScaler()),
    ])
    low_card = Pipeline([
        ("impute", SimpleImputer(strategy="constant", fill_value="Unknown")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=20)),
    ])
    high_card = Pipeline([
        ("impute", SimpleImputer(strategy="constant", fill_value="Unknown")),
        ("target", TargetEncoder(target_type="continuous", random_state=random_state)),
        ("scale", StandardScaler()),
    ])
    return ColumnTransformer([
        ("skew", skewed, skewed_features),
        ("num", numeric, other_numeric),
        ("bin", "passthrough", binary_features),
        ("low", low_card, low_card_categorical),
        ("high", high_card, high_card_categorical),
    ])

def evaluate(y_true, y_pred):
    pct_err = np.abs(y_pred - y_true) / y_true
    return {
        "R2": r2_score(y_true, y_pred),
        "MAPE %": pct_err.mean() * 100,
        "MdAPE %": np.median(pct_err) * 100,
        "MAE $": mean_absolute_error(y_true, y_pred),
        "Within 20 %": (pct_err <= 0.20).mean() * 100,
    }

def fit_and_score(train, eval_set):
    b = fit_outlier_bounds(train)
    tr, ev = apply_outlier_bounds(train, b), apply_outlier_bounds(eval_set, b)
    model = Pipeline([("prep", make_preprocessor()), ("lr", LinearRegression())])
    model.fit(tr[features], np.log(tr[target]))
    pred = np.exp(model.predict(ev[features]))
    return evaluate(ev[target].values, pred), len(tr), len(ev), model

In [14]:
months = sorted(clean["CloseMonth"].unique())
test_month = months[-1]
train_months = months[-1 - best_x : -1]

train = clean[clean["CloseMonth"].isin(train_months)]
test = clean[clean["CloseMonth"] == test_month]

bounds = fit_outlier_bounds(train)
train_f = apply_outlier_bounds(train, bounds)
test_f = apply_outlier_bounds(test, bounds)

print("Train:", train_months[0], "to", train_months[-1], f"| {len(train):,} -> {len(train_f):,} after outliers")
print("Test: ", test_month, f"| {len(test):,} -> {len(test_f):,} after outliers")
print({k: round(v, 2) for k, v in bounds.items()})

Train: 2025-09 to 2026-03 | 71,484 -> 70,414 after outliers
Test:  2026-04 | 12,455 -> 12,262 after outliers
{'price_lo': np.float64(184000.0), 'price_hi': np.float64(8953587.5), 'ppsf_lo': np.float64(138.73), 'ppsf_hi': np.float64(2316.67)}


In [15]:
baseline = Pipeline([("prep", make_preprocessor()), ("lr", LinearRegression())])
baseline.fit(train_f[features], np.log(train_f[target]))

train_pred = np.exp(baseline.predict(train_f[features]))
test_pred = np.exp(baseline.predict(test_f[features]))

results = pd.DataFrame({
    "train": evaluate(train_f[target].values, train_pred),
    "test (April)": evaluate(test_f[target].values, test_pred),
}).T
results["R2 (log)"] = [r2_score(np.log(train_f[target]), np.log(train_pred)),
                       r2_score(np.log(test_f[target]), np.log(test_pred))]
results.round(3)

,R2,MAPE %,MdAPE %,MAE $,Within 20 %,R2 (log)
train,0.818,15.168,11.409,195576.103,74.560,0.893
test (April),0.813,15.601,11.699,213024.688,73.414,0.890


In [16]:
def make_preprocessor_nolog():
    """Same as make_preprocessor, but sizes are NOT log-transformed."""
    numeric = Pipeline([
        ("impute", SimpleImputer(strategy="median", add_indicator=True)),
        ("scale", StandardScaler()),
    ])
    low_card = Pipeline([
        ("impute", SimpleImputer(strategy="constant", fill_value="Unknown")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=20)),
    ])
    high_card = Pipeline([
        ("impute", SimpleImputer(strategy="constant", fill_value="Unknown")),
        ("target", TargetEncoder(target_type="continuous", random_state=random_state)),
        ("scale", StandardScaler()),
    ])
    return ColumnTransformer([
        ("num", numeric, numeric_features),
        ("bin", "passthrough", binary_features),
        ("low", low_card, low_card_categorical),
        ("high", high_card, high_card_categorical),
    ])

y_true = test_f[target].values
comparison = {}

# A. Plain: raw price, raw sizes
m = Pipeline([("prep", make_preprocessor_nolog()), ("lr", LinearRegression())])
m.fit(train_f[features], train_f[target])
comparison["A. Plain linear"] = evaluate(y_true, m.predict(test_f[features]))

# B. Log price only
m = Pipeline([("prep", make_preprocessor_nolog()), ("lr", LinearRegression())])
m.fit(train_f[features], np.log(train_f[target]))
comparison["B. Log price only"] = evaluate(y_true, np.exp(m.predict(test_f[features])))

# C. Log-log: log price + log sizes
comparison["C. Log-log (baseline)"] = evaluate(y_true, test_pred)

pd.DataFrame(comparison).T.round(3)

,R2,MAPE %,MdAPE %,MAE $,Within 20 %
A. Plain linear,0.803,24.057,17.709,259315.804,55.244
B. Log price only,0.703,15.615,11.522,218907.663,74.034
C. Log-log (baseline),0.813,15.601,11.699,213024.688,73.414


In [17]:
def backtest(data, X, n_cutoffs=3):
    months = sorted(data["CloseMonth"].unique())
    rows = []
    for test_month in months[::-1][:n_cutoffs]:
        t = months.index(test_month)
        if t < X:
            print(f"Skipping {test_month}: needs {X} prior months, only {t} available")
            continue
        tr = data[data["CloseMonth"].isin(months[t - X:t])]
        te = data[data["CloseMonth"] == test_month]
        metrics, n_tr, n_te, _ = fit_and_score(tr, te)
        rows.append({"test month": test_month, "train rows": n_tr, "test rows": n_te, **metrics})
    return pd.DataFrame(rows).round(3)

backtest_results = backtest(clean, best_x)
backtest_results

,test month,train rows,test rows,R2,MAPE %,MdAPE %,MAE $,Within 20 %
0,2026-04,70414,12262,0.813,15.601,11.699,213024.688,73.414
1,2026-03,70280,11347,0.800,15.391,11.813,204797.950,73.914
2,2026-02,73357,8804,0.798,15.720,11.663,209351.074,73.853


In [18]:
res = test_f[[target, "CountyOrParish", "City", "LivingArea", "LotSqFt"]].copy()
res["predicted"] = test_pred
res["pct_err"] = (res["predicted"] - res[target]).abs() / res[target] * 100
res["signed_err"] = (res["predicted"] - res[target]) / res[target] * 100

res["price_band"] = pd.qcut(res[target], 5,
                            labels=["Q1 cheapest", "Q2", "Q3", "Q4", "Q5 priciest"])
by_band = res.groupby("price_band", observed=True).agg(
    homes=(target, "size"),
    min_price=(target, "min"),
    max_price=(target, "max"),
    MdAPE=("pct_err", "median"),
    MAPE=("pct_err", "mean"),
    bias=("signed_err", "median"),
).round(1)
print("Error by price band:")
display(by_band)

by_county = res.groupby("CountyOrParish").agg(
    homes=(target, "size"), MdAPE=("pct_err", "median"), bias=("signed_err", "median"))
print("Error by county (200+ test sales):")
by_county[by_county["homes"] >= 200].sort_values("MdAPE").round(1)

Error by price band:


,homes,min_price,max_price,MdAPE,MAPE,bias
price_band,,,,,,
Q1 cheapest,2487,185000.0,575000.0,11.4,17.5,4.3
Q2,2475,575003.0,800000.0,9.8,13.4,-0.1
Q3,2400,801663.0,1099000.0,9.9,12.9,-0.1
Q4,2449,1099900.0,1655000.0,11.8,14.7,-4.4
Q5 priciest,2451,1657900.0,8900000.0,16.7,19.5,-13.0


Error by county (200+ test sales):


,homes,MdAPE,bias
CountyOrParish,,,
San Bernardino,1228,10.1,-0.7
Riverside,1877,10.6,-0.8
Contra Costa,583,11.2,-3.3
Ventura,418,11.3,-2.7
Orange,1168,11.3,-3.1
San Diego,1435,12.6,-3.4
San Luis Obispo,215,13.0,0.1
Los Angeles,3013,13.1,-2.3
Santa Clara,392,13.2,-3.2


In [19]:
names = baseline.named_steps["prep"].get_feature_names_out()
coefs = pd.DataFrame({
    "coef": baseline.named_steps["lr"].coef_,
}, index=names)
coefs["effect %"] = (np.exp(coefs["coef"]) - 1) * 100
coefs = coefs.sort_values("effect %")

print("Biggest negative effects:")
display(coefs.head(10).round(2))
print("Biggest positive effects:")
coefs.tail(10).round(2)

Biggest negative effects:


,coef,effect %
low__CountyOrParish_Imperial,-0.45,-36.54
low__CountyOrParish_Tulare,-0.22,-19.66
low__CountyOrParish_Yuba,-0.19,-16.98
low__CountyOrParish_Riverside,-0.18,-16.73
low__CountyOrParish_infrequent_sklearn,-0.18,-16.64
low__CountyOrParish_Kern,-0.16,-15.11
low__CountyOrParish_Mariposa,-0.16,-14.97
low__CountyOrParish_Fresno,-0.15,-13.54
low__CountyOrParish_Madera,-0.13,-12.60
low__CountyOrParish_San Bernardino,-0.11,-10.73


Biggest positive effects:


,coef,effect %
low__CountyOrParish_Monterey,0.13,13.74
low__CountyOrParish_Sonoma,0.14,14.82
low__CountyOrParish_Santa Cruz,0.18,19.22
skew__LivingArea,0.19,20.64
low__CountyOrParish_Alameda,0.20,22.74
low__CountyOrParish_Santa Clara,0.27,30.50
low__CountyOrParish_San Mateo,0.29,33.56
high__PostalCode,0.31,36.16
low__CountyOrParish_Marin,0.32,38.23
low__CountyOrParish_San Francisco,0.35,42.47


In [20]:
res.sort_values("pct_err", ascending=False).head(10).round(1)

,ClosePrice,CountyOrParish,City,LivingArea,LotSqFt,predicted,pct_err,signed_err,price_band
102275,240000.0,San Diego,San Marcos,1680.0,NaN,1066517.8,344.4,344.4,Q1 cheapest
95517,296820.0,Santa Clara,Los Gatos,900.0,435600.0,1255269.0,322.9,322.9,Q1 cheapest
99744,355000.0,Santa Clara,Los Gatos,1248.0,31539.0,1317936.9,271.2,271.2,Q1 cheapest
105314,400000.0,Los Angeles,Sun Valley,2537.0,6287.0,1427501.7,256.9,256.9,Q1 cheapest
95075,525000.0,Los Angeles,Westlake Village,1776.0,4923.0,1687223.2,221.4,221.4,Q1 cheapest
106393,1400000.0,Los Angeles,Los Angeles,4790.0,23472.0,4154539.0,196.8,196.8,Q4
105812,200000.0,San Bernardino,Lake Arrowhead,1400.0,3375.0,588066.2,194.0,194.0,Q1 cheapest
99774,525000.0,Riverside,Indian Wells,1851.0,7405.0,1522413.4,190.0,190.0,Q1 cheapest
104344,190000.0,San Joaquin,Stockton,1294.0,8004.0,549087.6,189.0,189.0,Q1 cheapest
101213,689000.0,San Benito,Hollister,1684.0,3066.0,1979847.8,187.4,187.4,Q2


In [21]:
joblib.dump({
    "model": baseline,
    "bounds": bounds,
    "features": features,
    "train_months": train_months,
    "test_month": test_month,
}, f"{data_dir}/baseline_lr.joblib")

summary = results.loc[["test (April)"]].assign(model="Linear Regression", X=best_x)
summary.to_csv(f"{data_dir}/baseline_results.csv")
print("Saved baseline_lr.joblib and baseline_results.csv")

Saved baseline_lr.joblib and baseline_results.csv


Final Findings : Compared three linear setups on April. Plain linear: R² 0.80 but poor on typical homes (MdAPE 17.7%, 55% within 20%). Log price only: best MdAPE (11.5%) but R² drops to 0.70 from overpredicting very large homes. Log-log: R² 0.81, MdAPE 11.7%, 73% within 20%. Kept log-log: it ties on typical homes (the gap is within month-to-month noise) and avoids large misses.